In [1]:
"""
==============================================================================
 RANDOM FOREST — Baseline con registro de ejecuciones en Excel
==============================================================================
 Cada ejecución se AÑADE a un Excel acumulativo (no lo sobrescribe), para
 llevar una bitácora comparable de todas las configuraciones probadas.

 Hojas del Excel:
   - Ejecuciones    : historial; una fila por corrida (config + métricas)
   - Detalle_Folds  : desglose por fold de la última corrida
   - Importancia    : (preparada) significancia de cada variable
   - SHAP           : (preparada) resultados de SHAP value

 BLOQUES:
   1 Configuración  2 Carga/preparación  3 Validación
   4 Entrenamiento/evaluación  5 Reporte consola  6 Guardado Excel  7 Main
==============================================================================
"""

import os
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             median_absolute_error, r2_score)
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# Todo lo editable para cada experimento vive aquí.
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_RandomForest.xlsx",   # bitácora acumulativa
    "target"       : "Total_Parasitos_",     # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C",
        "Salinidad_psu",
        "Capa_Mezcla_m",
        "Altura_Mar_m",
        "Velocidad_Corriente_m_s",
    ],
    "usar_estacion": True,
    "col_estacion" : "Estacion",
    "validacion"   : "aleatoria",            # "temporal" o "aleatoria"
    "n_folds"      : 5,
    "random_state" : 42,
    "rf_params"    : {
        "random_state": 42,
        "n_jobs": -1,
        "max_depth": 20,
        "max_features": 1.0,
        "min_samples_split": 2,
        "n_estimators": 300,
    },
    "nota"         : "",                      # texto libre para identificar la corrida
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# Filtra registros con target, ordena por fecha y arma la matriz de features
# (con one-hot de estación si está activada).
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


# ============================================================================
# BLOQUE 3 — VALIDACIÓN
# Devuelve los pares (train, test) según la estrategia elegida.
# ============================================================================
def obtener_folds(config, X):
    if config["validacion"] == "temporal":
        splitter = TimeSeriesSplit(n_splits=config["n_folds"])
    elif config["validacion"] == "aleatoria":
        splitter = KFold(n_splits=config["n_folds"], shuffle=True,
                         random_state=config["random_state"])
    else:
        raise ValueError(f"Validación no reconocida: {config['validacion']}")
    return list(splitter.split(X))


# ============================================================================
# BLOQUE 4 — ENTRENAMIENTO Y EVALUACIÓN
# Entrena un RF por fold y calcula las cinco métricas de regresión.
# ============================================================================
def entrenar_evaluar(config, X, y, folds):
    resultados = []
    for i, (tr, te) in enumerate(folds):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        modelo = RandomForestRegressor(**config["rf_params"])
        modelo.fit(Xtr, ytr)
        pred = modelo.predict(Xte)

        rmse  = np.sqrt(mean_squared_error(yte, pred))
        media = yte.mean()
        resultados.append({
            "fold": i + 1, "n_train": len(tr), "n_test": len(te),
            "RMSE": rmse,
            "MAE": mean_absolute_error(yte, pred),
            "R2": r2_score(yte, pred),
            "MedAE": median_absolute_error(yte, pred),
            "RMSE_norm": rmse / media if media != 0 else np.nan,
        })
    return resultados


# ============================================================================
# BLOQUE 5 — REPORTE POR CONSOLA
# Imprime la tabla por fold y los promedios (lectura rápida en terminal).
# ============================================================================
def reportar(config, resultados, features):
    print("=" * 86)
    print("  RANDOM FOREST —", config["target"],
          "| estación:", "sí" if config["usar_estacion"] else "no",
          "| validación:", config["validacion"])
    print("=" * 86)
    print(f"  {'Fold':<6}{'n_test':<8}{'RMSE':<10}{'MAE':<10}{'R²':<10}"
          f"{'MedAE':<10}{'RMSE_norm':<12}")
    print("-" * 86)
    for r in resultados:
        print(f"  {r['fold']:<6}{r['n_test']:<8}{r['RMSE']:<10.4f}{r['MAE']:<10.4f}"
              f"{r['R2']:<10.4f}{r['MedAE']:<10.4f}{r['RMSE_norm']:<12.4f}")
    col = lambda k: np.array([r[k] for r in resultados])
    print("-" * 86)
    print(f"  {'MEDIA':<14}{col('RMSE').mean():<10.4f}{col('MAE').mean():<10.4f}"
          f"{col('R2').mean():<10.4f}{col('MedAE').mean():<10.4f}"
          f"{col('RMSE_norm').mean():<12.4f}")
    print("=" * 86)


# ============================================================================
# BLOQUE 6 — GUARDADO EN EXCEL (acumulativo)
# Si el Excel no existe, lo crea con sus hojas. Si existe, AÑADE la corrida
# a "Ejecuciones" y reemplaza "Detalle_Folds" con el desglose más reciente.
# Las hojas "Importancia" y "SHAP" se dejan preparadas para llenarse luego.
# ============================================================================
def guardar_excel(config, resultados, features):
    ruta = config["excel_salida"]
    col = lambda k: np.array([r[k] for r in resultados])

    # Fila resumen de esta ejecución (config + métricas promedio)
    fila = {
        "Fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "Modelo": "RandomForest",
        "Target": config["target"],
        "Estacion": "Sí" if config["usar_estacion"] else "No",
        "Validacion": config["validacion"],
        "N_folds": config["n_folds"],
        "N_features": len(features),
        "Hiperparametros": str(config["rf_params"]),
        "RMSE": round(col("RMSE").mean(), 4),
        "MAE": round(col("MAE").mean(), 4),
        "R2": round(col("R2").mean(), 4),
        "MedAE": round(col("MedAE").mean(), 4),
        "RMSE_norm": round(col("RMSE_norm").mean(), 4),
        "R2_desv": round(col("R2").std(), 4),
        "Nota": config["nota"],
    }

    encabezado_fill = PatternFill("solid", start_color="1F4E78")
    encabezado_font = Font(bold=True, color="FFFFFF", name="Arial")

    # Crear el archivo con sus hojas si aún no existe
    if not os.path.exists(ruta):
        wb = Workbook()
        ws = wb.active
        ws.title = "Ejecuciones"
        ws.append(list(fila.keys()))
        for c in ws[1]:
            c.fill = encabezado_fill; c.font = encabezado_font
            c.alignment = Alignment(horizontal="center")

        # Hoja de detalle por fold
        wb.create_sheet("Detalle_Folds")

        # Hoja preparada para importancia de variables
        ws_imp = wb.create_sheet("Importancia")
        ws_imp.append(["Fecha_ejecucion", "Target", "Variable",
                       "Importancia_RF", "Ranking"])
        for c in ws_imp[1]:
            c.fill = encabezado_fill; c.font = encabezado_font

        # Hoja preparada para SHAP
        ws_shap = wb.create_sheet("SHAP")
        ws_shap.append(["Fecha_ejecucion", "Target", "Variable",
                        "SHAP_medio_abs", "Direccion", "Ranking"])
        for c in ws_shap[1]:
            c.fill = encabezado_fill; c.font = encabezado_font

        wb.save(ruta)

    # Abrir el archivo y AÑADIR la nueva ejecución
    wb = load_workbook(ruta)
    wb["Ejecuciones"].append(list(fila.values()))

    # Reemplazar el detalle por fold con el de esta corrida
    ws_det = wb["Detalle_Folds"]
    ws_det.delete_rows(1, ws_det.max_row)   # limpiar lo anterior
    cabecera = ["Fold", "n_train", "n_test", "RMSE", "MAE",
                "R2", "MedAE", "RMSE_norm"]
    ws_det.append(cabecera)
    for c in ws_det[1]:
        c.fill = encabezado_fill; c.font = encabezado_font
    for r in resultados:
        ws_det.append([r["fold"], r["n_train"], r["n_test"],
                       round(r["RMSE"], 4), round(r["MAE"], 4),
                       round(r["R2"], 4), round(r["MedAE"], 4),
                       round(r["RMSE_norm"], 4)])

    # Ajuste simple de ancho de columnas en Ejecuciones
    ws_ej = wb["Ejecuciones"]
    for col_cells in ws_ej.columns:
        ancho = max(len(str(c.value)) for c in col_cells if c.value) + 2
        ws_ej.column_dimensions[col_cells[0].column_letter].width = min(ancho, 40)

    wb.save(ruta)
    print(f"\n[Excel] Ejecución registrada en: {ruta}")


# ============================================================================
# BLOQUE 7 — MAIN
# Orquesta el flujo completo: datos → folds → evaluación → consola → Excel.
# ============================================================================
def main():
    X, y, df, features = cargar_datos(CONFIG)
    folds = obtener_folds(CONFIG, X)
    resultados = entrenar_evaluar(CONFIG, X, y, folds)
    reportar(CONFIG, resultados, features)
    guardar_excel(CONFIG, resultados, features)


if __name__ == "__main__":
    main()

  RANDOM FOREST — Total_Parasitos_ | estación: sí | validación: aleatoria
  Fold  n_test  RMSE      MAE       R²        MedAE     RMSE_norm   
--------------------------------------------------------------------------------------
  1     476     1.9118    1.1734    0.7475    0.6323    0.7263      
  2     476     2.0351    1.1982    0.7497    0.5516    0.7502      
  3     476     1.7286    1.0506    0.7665    0.5561    0.6819      
  4     475     1.7562    1.0933    0.7335    0.6148    0.7364      
  5     475     1.7599    1.1009    0.7352    0.5844    0.7260      
--------------------------------------------------------------------------------------
  MEDIA         1.8383    1.1233    0.7465    0.5879    0.7242      

[Excel] Ejecución registrada en: Registro_RandomForest.xlsx


In [ ]:
"""
==============================================================================
 RANDOM FOREST — Optimización de hiperparámetros con GridSearchCV
==============================================================================
 Búsqueda en grilla guiada por MAE (robusto al sesgo del target).
 La combinación ganadora se reevalúa reportando las CINCO métricas y se
 registra en el Excel acumulativo, en una hoja propia "Optimizacion".

 Validación interna de la grilla: KFold aleatorio (coherente con la
 validación honesta usada en todo el estudio).

 BLOQUES:
   1 Configuración  2 Carga/preparación  3 Grilla + GridSearchCV
   4 Reevaluación con 5 métricas  5 Reporte consola  6 Guardado Excel  7 Main
==============================================================================
"""

import os
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             median_absolute_error, r2_score, make_scorer)
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# Datos, target, features, y la GRILLA de hiperparámetros a explorar.
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_RandomForest.xlsx",
    "target"       : "Total_Parasitos_",   # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    "usar_estacion": False,
    "col_estacion" : "Estacion",
    "n_folds"      : 5,
    "random_state" : 42,

    # Grilla propuesta: 3 x 3 x 3 x 2 = 54 combinaciones x 5 folds = 270 fits
    "grilla" : {
        "n_estimators"     : [100, 300, 500],
        "max_depth"        : [10, 20, None, 50],
        "min_samples_split": [2, 5, 10],
        "max_features"     : ["sqrt", 1.0],
    },
    "nota"         : "GridSearch guiado por MAE",
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


# ============================================================================
# BLOQUE 3 — GRILLA + GridSearchCV (guiado por MAE)
# Busca la mejor combinación minimizando el MAE en validación cruzada.
# ============================================================================
def optimizar(config, X, y):
    cv = KFold(n_splits=config["n_folds"], shuffle=True,
               random_state=config["random_state"])

    # scoring: GridSearchCV maximiza, por eso MAE va como "neg" (negativo)
    scorer_mae = make_scorer(mean_absolute_error, greater_is_better=False)

    base = RandomForestRegressor(random_state=config["random_state"], n_jobs=-1)

    grid = GridSearchCV(
        estimator=base,
        param_grid=config["grilla"],
        scoring=scorer_mae,     # guía la decisión por MAE
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X, y)
    return grid, cv


# ============================================================================
# BLOQUE 4 — REEVALUACIÓN DEL MEJOR MODELO CON LAS 5 MÉTRICAS
# Con los mejores parámetros, predice por validación cruzada y calcula
# las cinco métricas sobre esas predicciones out-of-fold.
# ============================================================================
def reevaluar(config, X, y, mejores_params, cv):
    modelo = RandomForestRegressor(
        random_state=config["random_state"], n_jobs=-1, **mejores_params
    )
    # Predicciones fuera de fold (cada dato predicho por un modelo que no lo vio)
    pred = cross_val_predict(modelo, X, y, cv=cv, n_jobs=-1)

    rmse  = np.sqrt(mean_squared_error(y, pred))
    media = y.mean()
    metricas = {
        "RMSE": rmse,
        "MAE": mean_absolute_error(y, pred),
        "R2": r2_score(y, pred),
        "MedAE": median_absolute_error(y, pred),
        "RMSE_norm": rmse / media if media != 0 else np.nan,
    }
    return metricas


# ============================================================================
# BLOQUE 5 — REPORTE POR CONSOLA
# ============================================================================
def reportar(config, grid, metricas, features):
    print("=" * 80)
    print("  RANDOM FOREST — OPTIMIZACIÓN GridSearchCV (guía: MAE)")
    print("=" * 80)
    print(f"  Target        : {config['target']}")
    print(f"  Estación      : {'Sí' if config['usar_estacion'] else 'No'}")
    print(f"  Features      : {len(features)}")
    print(f"  Combinaciones : {len(grid.cv_results_['params'])}  "
          f"(x {config['n_folds']} folds)")
    print(f"\n  Mejores hiperparámetros encontrados:")
    for k, v in grid.best_params_.items():
        print(f"     {k:<20}: {v}")
    print(f"\n  MAE de la búsqueda (CV) : {-grid.best_score_:.4f}")
    print(f"\n  Métricas del mejor modelo (predicción out-of-fold):")
    print(f"     RMSE      : {metricas['RMSE']:.4f}")
    print(f"     MAE       : {metricas['MAE']:.4f}")
    print(f"     R²        : {metricas['R2']:.4f}")
    print(f"     MedAE     : {metricas['MedAE']:.4f}")
    print(f"     RMSE_norm : {metricas['RMSE_norm']:.4f}")
    print("=" * 80)


# ============================================================================
# BLOQUE 6 — GUARDADO EN EXCEL (hoja "Optimizacion", acumulativa)
# ============================================================================
def guardar_excel(config, grid, metricas, features):
    ruta = config["excel_salida"]
    enc_fill = PatternFill("solid", start_color="7030A0")   # morado optimización
    enc_font = Font(bold=True, color="FFFFFF", name="Arial")

    cabecera = ["Fecha_ejecucion", "Modelo", "Target", "Estacion",
                "N_features", "Mejores_parametros", "MAE_CV",
                "RMSE", "MAE", "R2", "MedAE", "RMSE_norm", "Nota"]
    fila = [
        datetime.now().strftime("%Y-%m-%d %H:%M"), "RandomForest",
        config["target"], "Sí" if config["usar_estacion"] else "No",
        len(features), str(grid.best_params_), round(-grid.best_score_, 4),
        round(metricas["RMSE"], 4), round(metricas["MAE"], 4),
        round(metricas["R2"], 4), round(metricas["MedAE"], 4),
        round(metricas["RMSE_norm"], 4), config["nota"],
    ]

    # Crear archivo base si no existe (con la hoja de Ejecuciones vacía mínima)
    if not os.path.exists(ruta):
        wb = Workbook(); wb.active.title = "Ejecuciones"; wb.save(ruta)

    wb = load_workbook(ruta)
    if "Optimizacion" not in wb.sheetnames:
        ws = wb.create_sheet("Optimizacion")
        ws.append(cabecera)
        for c in ws[1]:
            c.fill = enc_fill; c.font = enc_font
            c.alignment = Alignment(horizontal="center")
    ws = wb["Optimizacion"]
    ws.append(fila)

    for col_cells in ws.columns:
        ancho = max(len(str(c.value)) for c in col_cells if c.value) + 2
        ws.column_dimensions[col_cells[0].column_letter].width = min(ancho, 45)

    wb.save(ruta)
    print(f"\n[Excel] Optimización registrada en: {ruta} (hoja 'Optimizacion')")


# ============================================================================
# BLOQUE 7 — MAIN
# ============================================================================
def main():
    X, y, df, features = cargar_datos(CONFIG)
    grid, cv = optimizar(CONFIG, X, y)
    metricas = reevaluar(CONFIG, X, y, grid.best_params_, cv)
    reportar(CONFIG, grid, metricas, features)
    guardar_excel(CONFIG, grid, metricas, features)


if __name__ == "__main__":
    main()

Fitting 5 folds for each of 72 candidates, totalling 360 fits


In [5]:
"""
==============================================================================
 RANDOM FOREST — Análisis SHAP (interpretación, OE3)
==============================================================================
 Calcula los valores SHAP sobre un Random Forest entrenado con los
 hiperparámetros óptimos, para identificar el impacto de cada variable
 física sobre la prevalencia (magnitud + dirección).

 Requiere: pip install shap

 Genera:
   - Gráfico beeswarm  (magnitud + dirección por observación)  → el principal
   - Gráfico bar        (ranking de importancia media |SHAP|)
   - Hoja "SHAP" del Excel rellenada con SHAP medio absoluto y dirección
   - (opcional) SHAP comparativo por región para el OE4

 NOTA: SHAP es post-hoc. El modelo se entrena primero; los valores SHAP se
 calculan sobre un conjunto de test (no sobre el de entrenamiento), para que
 la interpretación refleje cómo generaliza el modelo.

 BLOQUES:
   1 Configuración  2 Carga/preparación  3 Entrenamiento
   4 Cálculo SHAP   5 Gráficos          6 Guardado Excel  7 Main
==============================================================================
"""

import os
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill
import shap
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# Incluye los hiperparámetros ÓPTIMOS según target y estación.
# Cambia "target" y "usar_estacion" para cada una de las 4 combinaciones.
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_RandomForest.xlsx",
    "carpeta_img"  : "SHAP_RandomForest",       # donde guardar los gráficos
    "target"       : "Hembras_Ovigeras",   # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    "usar_estacion": False,
    "col_estacion" : "Estacion",
    "test_size"    : 0.25,        # proporción para calcular SHAP
    "random_state" : 42,
    "shap_por_region": False,     # True = genera comparativo por región (OE4)

    # Hiperparámetros óptimos por combinación (target, estacion).
    # El código elige el set correcto automáticamente.
    "hiperparametros": {
        ("Total_Parasitos_", False): {"max_depth": None, "max_features": 1.0,
                                       "min_samples_split": 2, "n_estimators": 500},
        ("Total_Parasitos_", True) : {"max_depth": 20, "max_features": 1.0,
                                       "min_samples_split": 2, "n_estimators": 300},
        ("Hembras_Ovigeras", False): {"max_depth": None, "max_features": 1.0,
                                       "min_samples_split": 2, "n_estimators": 300},
        ("Hembras_Ovigeras", True) : {"max_depth": 20, "max_features": 1.0,
                                       "min_samples_split": 2, "n_estimators": 300},
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    return X, y, df, features


# ============================================================================
# BLOQUE 3 — ENTRENAMIENTO (con hiperparámetros óptimos)
# ============================================================================
def entrenar(config, X, y):
    clave = (config["target"], config["usar_estacion"])
    params = config["hiperparametros"][clave]
    print(f"  Hiperparámetros usados: {params}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config["test_size"],
        random_state=config["random_state"]
    )
    modelo = RandomForestRegressor(
        random_state=config["random_state"], n_jobs=-1, **params
    )
    modelo.fit(X_train, y_train)
    return modelo, X_train, X_test, y_test


# ============================================================================
# BLOQUE 4 — CÁLCULO SHAP (TreeExplainer, eficiente para árboles)
# ============================================================================
def calcular_shap(modelo, X_test):
    explainer = shap.TreeExplainer(modelo)
    shap_values = explainer.shap_values(X_test)
    # Importancia media absoluta por variable (ranking global)
    importancia = np.abs(shap_values).mean(axis=0)
    # Dirección: correlación entre valor de la feature y su SHAP
    direcciones = []
    for j in range(X_test.shape[1]):
        corr = np.corrcoef(X_test.iloc[:, j], shap_values[:, j])[0, 1]
        direcciones.append("positiva" if corr > 0 else "negativa")
    return shap_values, importancia, direcciones


# ============================================================================
# BLOQUE 5 — GRÁFICOS (beeswarm + barra)
# ============================================================================
def graficos(config, shap_values, X_test, features):
    os.makedirs(config["carpeta_img"], exist_ok=True)
    sufijo = f"{config['target']}_{'conEst' if config['usar_estacion'] else 'sinEst'}"

    # Beeswarm — magnitud + dirección (el gráfico principal del OE3)
    plt.figure()
    shap.summary_plot(shap_values, X_test, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(config["carpeta_img"], f"beeswarm_{sufijo}.png"),
                dpi=150, bbox_inches="tight")
    plt.close()

    # Bar — ranking de importancia
    plt.figure()
    shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(config["carpeta_img"], f"bar_{sufijo}.png"),
                dpi=150, bbox_inches="tight")
    plt.close()

    print(f"  Gráficos guardados en: {config['carpeta_img']}/ (sufijo {sufijo})")


# ============================================================================
# BLOQUE 6 — GUARDADO EN EXCEL (hoja "SHAP", acumulativa)
# ============================================================================
def guardar_excel(config, importancia, direcciones, features):
    ruta = config["excel_salida"]
    orden = np.argsort(importancia)[::-1]   # de mayor a menor importancia

    enc_fill = PatternFill("solid", start_color="C55A11")  # naranja SHAP
    enc_font = Font(bold=True, color="FFFFFF", name="Arial")

    if not os.path.exists(ruta):
        wb = Workbook(); wb.active.title = "Ejecuciones"; wb.save(ruta)
    wb = load_workbook(ruta)
    if "SHAP" not in wb.sheetnames:
        ws = wb.create_sheet("SHAP")
        ws.append(["Fecha_ejecucion", "Target", "Estacion", "Variable",
                   "SHAP_medio_abs", "Direccion", "Ranking"])
        for c in ws[1]:
            c.fill = enc_fill; c.font = enc_font
    ws = wb["SHAP"]

    fecha = datetime.now().strftime("%Y-%m-%d %H:%M")
    est = "Sí" if config["usar_estacion"] else "No"
    for rank, idx in enumerate(orden, start=1):
        ws.append([fecha, config["target"], est, features[idx],
                   round(float(importancia[idx]), 5), direcciones[idx], rank])

    wb.save(ruta)
    print(f"  SHAP registrado en: {ruta} (hoja 'SHAP')")


# ============================================================================
# BLOQUE 7 — MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("  SHAP — RANDOM FOREST")
    print(f"  Target: {CONFIG['target']} | Estación: "
          f"{'Sí' if CONFIG['usar_estacion'] else 'No'}")
    print("=" * 70)

    X, y, df, features = cargar_datos(CONFIG)
    modelo, X_train, X_test, y_test = entrenar(CONFIG, X, y)
    shap_values, importancia, direcciones = calcular_shap(modelo, X_test)

    print("\n  Ranking de importancia (SHAP medio absoluto):")
    orden = np.argsort(importancia)[::-1]
    for rank, idx in enumerate(orden, start=1):
        print(f"    {rank}. {features[idx]:<24} "
              f"{importancia[idx]:.4f}  ({direcciones[idx]})")

    graficos(CONFIG, shap_values, X_test, features)
    guardar_excel(CONFIG, importancia, direcciones, features)
    print("\n  Completado.")


if __name__ == "__main__":
    main()

  SHAP — RANDOM FOREST
  Target: Hembras_Ovigeras | Estación: No
  Hiperparámetros usados: {'max_depth': None, 'max_features': 1.0, 'min_samples_split': 2, 'n_estimators': 300}

  Ranking de importancia (SHAP medio absoluto):
    1. Temperatura_C            0.4995  (positiva)
    2. Salinidad_psu            0.1850  (positiva)
    3. Capa_Mezcla_m            0.1206  (positiva)
    4. Altura_Mar_m             0.1024  (negativa)
    5. Velocidad_Corriente_m_s  0.0357  (positiva)
  Gráficos guardados en: SHAP_RandomForest/ (sufijo Hembras_Ovigeras_sinEst)
  SHAP registrado en: Registro_RandomForest.xlsx (hoja 'SHAP')

  Completado.


In [1]:
"""
==============================================================================
 RANDOM FOREST — SHAP por región + Perfiles de infestación  (OE3 + OE4)
==============================================================================
 Tres análisis integrados:
   A) SHAP GLOBAL          → ranking de importancia general (OE3)
   B) SHAP POR REGIÓN      → compara Aysén vs Magallanes (OE4)
   C) PERFILES INFESTACIÓN → rangos de cada variable física por nivel
                             (bajo/medio/alto) y región, MÁS los conteos
                             promedio de cada estadio (juveniles, adultos,
                             hembras ovígeras) como columnas descriptivas.

 La estación se incorpora como flag (usar_estacion) para generar las dos
 versiones del SHAP, igual que en los demás códigos. Los estadios NO son
 features del modelo: solo describen la composición del parásito en cada
 nivel/región.

 Requiere: pip install shap

 BLOQUES:
   1 Configuración   2 Carga/preparación   3 Entrenamiento
   4 SHAP global     5 SHAP por región     6 Perfiles de infestación
   7 Gráficos        8 Guardado Excel       9 Main
==============================================================================
"""

import os
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill
import shap
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_RandomForest.xlsx",
    "carpeta_img"  : "SHAP_RF_region",
    "target"       : "Hembras_Ovigeras",   # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    # Estadios que se muestran como columnas descriptivas en los perfiles
    "estadios"     : ["Juveniles", "Adultos_Moviles", "Hembras_Ovigeras"],
    "col_region"   : "Region",
    "col_estacion" : "Estacion",
    "usar_estacion": True,        # True = añade estación al modelo SHAP
    "test_size"    : 0.25,
    "random_state" : 42,

    # Hiperparámetros óptimos por (target, estacion)
    "hiperparametros": {
        ("Total_Parasitos_", False): {"max_depth": None, "max_features": 1.0,
                                      "min_samples_split": 2, "n_estimators": 500},
        ("Total_Parasitos_", True) : {"max_depth": 20, "max_features": 1.0,
                                      "min_samples_split": 2, "n_estimators": 300},
        ("Hembras_Ovigeras", False): {"max_depth": None, "max_features": 1.0,
                                      "min_samples_split": 2, "n_estimators": 300},
        ("Hembras_Ovigeras", True) : {"max_depth": 20, "max_features": 1.0,
                                      "min_samples_split": 2, "n_estimators": 300},
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    region = df[config["col_region"]].copy()
    return X, y, region, df, features


# ============================================================================
# BLOQUE 3 — ENTRENAMIENTO
# ============================================================================
def entrenar(config, X, y, region):
    clave = (config["target"], config["usar_estacion"])
    params = config["hiperparametros"][clave]
    print(f"  Hiperparámetros: {params}")
    idx = np.arange(len(X))
    Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(
        X, y, idx, test_size=config["test_size"],
        random_state=config["random_state"]
    )
    modelo = RandomForestRegressor(random_state=config["random_state"],
                                   n_jobs=-1, **params)
    modelo.fit(Xtr, ytr)
    region_te = region.iloc[idx_te].values
    return modelo, Xte, yte, region_te


# ============================================================================
# BLOQUE 4 — SHAP GLOBAL
# ============================================================================
def shap_global(modelo, X_test, features):
    explainer = shap.TreeExplainer(modelo)
    shap_values = explainer.shap_values(X_test)
    importancia = np.abs(shap_values).mean(axis=0)
    direcciones = []
    for j in range(X_test.shape[1]):
        corr = np.corrcoef(X_test.iloc[:, j], shap_values[:, j])[0, 1]
        direcciones.append("positiva" if corr > 0 else "negativa")
    return explainer, shap_values, importancia, direcciones


# ============================================================================
# BLOQUE 5 — SHAP POR REGIÓN (OE4)
# ============================================================================
def shap_por_region(explainer, X_test, region_te, features):
    resultados = {}
    for reg in np.unique(region_te):
        mask = region_te == reg
        if mask.sum() < 10:
            continue
        sv = explainer.shap_values(X_test[mask])
        resultados[reg] = np.abs(sv).mean(axis=0)
    return resultados


# ============================================================================
# BLOQUE 6 — PERFILES DE INFESTACIÓN (bajo/medio/alto)
# Para cada nivel y región: rangos de variables físicas (P10–med–P90) +
# promedio de cada estadio del parásito (columnas descriptivas).
# ============================================================================
def perfiles_infestacion(config, X, y, region, df):
    p33 = y.quantile(0.33)
    p66 = y.quantile(0.66)

    def nivel(v):
        if v <= p33:   return "bajo"
        elif v <= p66: return "medio"
        else:          return "alto"
    niveles = y.apply(nivel)

    # IMPORTANTE: los perfiles usan SOLO las variables físicas continuas,
    # tomadas de config["features"] (las cinco base). Nunca las columnas de
    # estación (one-hot binarias), porque calcular percentiles sobre 0/1 no
    # tiene sentido. Se leen desde df, que siempre tiene las columnas físicas.
    fisicas = config["features"]   # las cinco físicas continuas (fijas)

    def resumen(sub_idx, etiqueta):
        filas = []
        for nv in ["bajo", "medio", "alto"]:
            m = (niveles == nv) & sub_idx
            if m.sum() == 0:
                continue
            fila = {"Grupo": etiqueta, "Nivel": nv, "n": int(m.sum())}
            # Rangos de variables físicas continuas (P10 – mediana – P90)
            for f in fisicas:
                fila[f"{f}_p10"] = round(df.loc[m, f].quantile(0.10), 2)
                fila[f"{f}_med"] = round(df.loc[m, f].median(), 2)
                fila[f"{f}_p90"] = round(df.loc[m, f].quantile(0.90), 2)
            # Conteo promedio de cada estadio (descriptivo, no del modelo)
            for est in config["estadios"]:
                fila[f"{est}_prom"] = round(df.loc[m, est].mean(), 2)
            filas.append(fila)
        return filas

    todo = pd.Series(True, index=y.index)
    perfil = resumen(todo, "GLOBAL")
    for reg in region.unique():
        perfil += resumen(region == reg, reg)
    return perfil, (p33, p66)


# ============================================================================
# BLOQUE 7 — GRÁFICOS
# ============================================================================
def graficos(config, shap_values, X_test, features, shap_reg):
    os.makedirs(config["carpeta_img"], exist_ok=True)
    suf = f"{config['target']}_{'conEst' if config['usar_estacion'] else 'sinEst'}"

    plt.figure()
    shap.summary_plot(shap_values, X_test, show=False)
    plt.tight_layout()
    plt.savefig(f"{config['carpeta_img']}/beeswarm_{suf}.png", dpi=150,
                bbox_inches="tight"); plt.close()

    if shap_reg:
        regiones = list(shap_reg.keys())
        x = np.arange(len(features))
        w = 0.8 / len(regiones)
        plt.figure(figsize=(9, 5))
        for i, reg in enumerate(regiones):
            plt.barh(x + i*w, shap_reg[reg], height=w, label=reg)
        plt.yticks(x + w*(len(regiones)-1)/2, features)
        plt.xlabel("SHAP medio absoluto")
        plt.title(f"SHAP por región — {suf}")
        plt.legend(); plt.tight_layout()
        plt.savefig(f"{config['carpeta_img']}/region_{suf}.png", dpi=150,
                    bbox_inches="tight"); plt.close()
    print(f"  Gráficos en: {config['carpeta_img']}/")


# ============================================================================
# BLOQUE 8 — GUARDADO EN EXCEL
# ============================================================================
def guardar_excel(config, shap_reg, features, perfil):
    ruta = config["excel_salida"]
    if not os.path.exists(ruta):
        wb = Workbook(); wb.active.title = "Ejecuciones"; wb.save(ruta)
    wb = load_workbook(ruta)
    fecha = datetime.now().strftime("%Y-%m-%d %H:%M")
    est_txt = "Sí" if config["usar_estacion"] else "No"
    fill = PatternFill("solid", start_color="2E75B6")
    font = Font(bold=True, color="FFFFFF")

    # SHAP por región
    if "SHAP_Region" not in wb.sheetnames:
        ws = wb.create_sheet("SHAP_Region")
        ws.append(["Fecha", "Target", "Estacion", "Region", "Variable",
                   "SHAP_medio_abs", "Ranking"])
        for c in ws[1]: c.fill = fill; c.font = font
    ws = wb["SHAP_Region"]
    for reg, imp in shap_reg.items():
        for rank, idx in enumerate(np.argsort(imp)[::-1], 1):
            ws.append([fecha, config["target"], est_txt, reg, features[idx],
                       round(float(imp[idx]), 5), rank])

    # Perfiles de infestación (con estadios)
    if "Perfiles_Infestacion" not in wb.sheetnames:
        ws2 = wb.create_sheet("Perfiles_Infestacion")
        cab = ["Fecha", "Target", "Grupo", "Nivel", "n"]
        for f in config["features"]:
            cab += [f"{f}_p10", f"{f}_med", f"{f}_p90"]
        for est in config["estadios"]:
            cab += [f"{est}_prom"]
        ws2.append(cab)
        for c in ws2[1]: c.fill = fill; c.font = font
    ws2 = wb["Perfiles_Infestacion"]
    for fila in perfil:
        row = [fecha, config["target"], fila["Grupo"], fila["Nivel"], fila["n"]]
        for f in config["features"]:
            row += [fila[f"{f}_p10"], fila[f"{f}_med"], fila[f"{f}_p90"]]
        for est in config["estadios"]:
            row += [fila[f"{est}_prom"]]
        ws2.append(row)

    wb.save(ruta)
    print(f"  Registrado en Excel: hojas SHAP_Region y Perfiles_Infestacion")


# ============================================================================
# BLOQUE 9 — MAIN
# ============================================================================
def main():
    print("=" * 70)
    print(f"  SHAP REGIÓN + PERFILES — RF — {CONFIG['target']} — "
          f"estación: {'Sí' if CONFIG['usar_estacion'] else 'No'}")
    print("=" * 70)

    X, y, region, df, features = cargar_datos(CONFIG)
    modelo, X_test, y_test, region_te = entrenar(CONFIG, X, y, region)

    explainer, shap_values, importancia, direcciones = shap_global(
        modelo, X_test, features)
    print("\n  SHAP GLOBAL (ranking):")
    for rank, idx in enumerate(np.argsort(importancia)[::-1], 1):
        print(f"    {rank}. {features[idx]:<24} "
              f"{importancia[idx]:.4f} ({direcciones[idx]})")

    shap_reg = shap_por_region(explainer, X_test, region_te, features)
    print("\n  SHAP POR REGIÓN:")
    for reg, imp in shap_reg.items():
        orden = np.argsort(imp)[::-1]
        print(f"    {reg}:")
        for rank, idx in enumerate(orden, 1):
            print(f"        {rank}. {features[idx]:<22} {imp[idx]:.4f}")

    perfil, (p33, p66) = perfiles_infestacion(CONFIG, X, y, region, df)
    print(f"\n  PERFILES (bajo<={p33:.2f}, medio<={p66:.2f}, alto>{p66:.2f}):")
    print("  Estadios promedio por nivel (ej. región Magallanes):")
    for fila in perfil:
        if fila["Grupo"] == "Magallanes":
            ests = " ".join(f"{e}={fila[f'{e}_prom']}" for e in CONFIG["estadios"])
            print(f"    {fila['Nivel']:<6} n={fila['n']:<5} {ests}")

    graficos(CONFIG, shap_values, X_test, features, shap_reg)
    guardar_excel(CONFIG, shap_reg, features, perfil)
    print("\n  Completado.")


if __name__ == "__main__":
    main()

  SHAP REGIÓN + PERFILES — RF — Hembras_Ovigeras — estación: Sí
  Hiperparámetros: {'max_depth': 20, 'max_features': 1.0, 'min_samples_split': 2, 'n_estimators': 300}

  SHAP GLOBAL (ranking):
    1. Temperatura_C            0.5219 (positiva)
    2. Est_Invierno             0.2556 (positiva)
    3. Salinidad_psu            0.2413 (positiva)
    4. Est_Primavera            0.1429 (positiva)
    5. Altura_Mar_m             0.0680 (negativa)
    6. Capa_Mezcla_m            0.0585 (positiva)
    7. Velocidad_Corriente_m_s  0.0246 (negativa)
    8. Est_Otoño                0.0189 (positiva)
    9. Est_Verano               0.0180 (negativa)

  SHAP POR REGIÓN:
    Aysen:
        1. Temperatura_C          0.4378
        2. Est_Invierno           0.2779
        3. Salinidad_psu          0.2768
        4. Est_Primavera          0.1638
        5. Altura_Mar_m           0.0768
        6. Capa_Mezcla_m          0.0686
        7. Velocidad_Corriente_m_s 0.0289
        8. Est_Otoño              0.01

#Lecturas de tiempo

In [ ]:
"""
==============================================================================
 RANDOM FOREST — SHAP por región + Perfiles de infestación  (OE3 + OE4)
==============================================================================
 Tres análisis integrados:
   A) SHAP GLOBAL          → ranking de importancia general (OE3)
   B) SHAP POR REGIÓN      → compara Aysén vs Magallanes (OE4)
   C) PERFILES INFESTACIÓN → rangos de cada variable física por nivel
                             (bajo/medio/alto) y región, MÁS los conteos
                             promedio de cada estadio (juveniles, adultos,
                             hembras ovígeras) como columnas descriptivas.

 La estación se incorpora como flag (usar_estacion) para generar las dos
 versiones del SHAP, igual que en los demás códigos. Los estadios NO son
 features del modelo: solo describen la composición del parásito en cada
 nivel/región.

 Requiere: pip install shap

 BLOQUES:
   1 Configuración   2 Carga/preparación   3 Entrenamiento
   4 SHAP global     5 SHAP por región     6 Perfiles de infestación
   7 Gráficos        8 Guardado Excel       9 Main
==============================================================================
"""

import os
import time
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill
import shap
import warnings
warnings.filterwarnings('ignore')


# ============================================================================
# BLOQUE 1 — CONFIGURACIÓN
# ============================================================================
CONFIG = {
    "ruta_csv"     : "data/Dataset_Maestro_Amplificado_ML.csv",
    "excel_salida" : "Registro_RandomForest.xlsx",
    "carpeta_img"  : "SHAP_RF_region",
    "target"       : "Total_Parasitos_",   # "Hembras_Ovigeras" o "Total_Parasitos_"
    "features"     : [
        "Temperatura_C", "Salinidad_psu", "Capa_Mezcla_m",
        "Altura_Mar_m", "Velocidad_Corriente_m_s",
    ],
    # Estadios que se muestran como columnas descriptivas en los perfiles
    "estadios"     : ["Juveniles", "Adultos_Moviles", "Hembras_Ovigeras"],
    "col_region"   : "Region",
    "col_estacion" : "Estacion",
    "usar_estacion": False,        # True = añade estación al modelo SHAP
    "test_size"    : 0.25,
    "random_state" : 42,

    # Hiperparámetros óptimos por (target, estacion)
    "hiperparametros": {
        ("Total_Parasitos_", False): {"max_depth": None, "max_features": 1.0,
                                      "min_samples_split": 2, "n_estimators": 500},
        ("Total_Parasitos_", True) : {"max_depth": 20, "max_features": 1.0,
                                      "min_samples_split": 2, "n_estimators": 300},
        ("Hembras_Ovigeras", False): {"max_depth": None, "max_features": 1.0,
                                      "min_samples_split": 2, "n_estimators": 300},
        ("Hembras_Ovigeras", True) : {"max_depth": 20, "max_features": 1.0,
                                      "min_samples_split": 2, "n_estimators": 300},
    },
}


# ============================================================================
# BLOQUE 2 — CARGA Y PREPARACIÓN
# ============================================================================
def cargar_datos(config):
    df = pd.read_csv(config["ruta_csv"])
    df["Fecha"] = pd.to_datetime(df["Fecha"])
    df = df[df[config["target"]].notna()].copy()
    df = df.sort_values("Fecha").reset_index(drop=True)

    features = list(config["features"])
    if config["usar_estacion"]:
        dummies = pd.get_dummies(df[config["col_estacion"]], prefix="Est")
        df = pd.concat([df, dummies], axis=1)
        features += list(dummies.columns)

    X = df[features].copy()
    y = df[config["target"]].copy()
    region = df[config["col_region"]].copy()
    return X, y, region, df, features


# ============================================================================
# BLOQUE 3 — ENTRENAMIENTO
# ============================================================================
def entrenar(config, X, y, region):
    clave = (config["target"], config["usar_estacion"])
    params = config["hiperparametros"][clave]
    print(f"  Hiperparámetros: {params}")
    idx = np.arange(len(X))
    Xtr, Xte, ytr, yte, idx_tr, idx_te = train_test_split(
        X, y, idx, test_size=config["test_size"],
        random_state=config["random_state"]
    )
    modelo = RandomForestRegressor(random_state=config["random_state"],
                                   n_jobs=-1, **params)
    # Medición del tiempo de entrenamiento (solo el .fit del modelo óptimo).
    # Se excluye deliberadamente la carga de datos, el cálculo de SHAP y el
    # guardado en Excel, para reflejar únicamente el costo de entrenamiento.
    t_inicio = time.perf_counter()
    modelo.fit(Xtr, ytr)
    tiempo_fit = time.perf_counter() - t_inicio
    print(f"  Tiempo de entrenamiento (.fit): {tiempo_fit:.4f} s")
    region_te = region.iloc[idx_te].values
    return modelo, Xte, yte, region_te, tiempo_fit


# ============================================================================
# BLOQUE 4 — SHAP GLOBAL
# ============================================================================
def shap_global(modelo, X_test, features):
    explainer = shap.TreeExplainer(modelo)
    shap_values = explainer.shap_values(X_test)
    importancia = np.abs(shap_values).mean(axis=0)
    direcciones = []
    for j in range(X_test.shape[1]):
        corr = np.corrcoef(X_test.iloc[:, j], shap_values[:, j])[0, 1]
        direcciones.append("positiva" if corr > 0 else "negativa")
    return explainer, shap_values, importancia, direcciones


# ============================================================================
# BLOQUE 5 — SHAP POR REGIÓN (OE4)
# ============================================================================
def shap_por_region(explainer, X_test, region_te, features):
    resultados = {}
    for reg in np.unique(region_te):
        mask = region_te == reg
        if mask.sum() < 10:
            continue
        sv = explainer.shap_values(X_test[mask])
        resultados[reg] = np.abs(sv).mean(axis=0)
    return resultados


# ============================================================================
# BLOQUE 6 — PERFILES DE INFESTACIÓN (bajo/medio/alto)
# Para cada nivel y región: rangos de variables físicas (P10–med–P90) +
# promedio de cada estadio del parásito (columnas descriptivas).
# ============================================================================
def perfiles_infestacion(config, X, y, region, df):
    p33 = y.quantile(0.33)
    p66 = y.quantile(0.66)

    def nivel(v):
        if v <= p33:   return "bajo"
        elif v <= p66: return "medio"
        else:          return "alto"
    niveles = y.apply(nivel)

    # IMPORTANTE: los perfiles usan SOLO las variables físicas continuas,
    # tomadas de config["features"] (las cinco base). Nunca las columnas de
    # estación (one-hot binarias), porque calcular percentiles sobre 0/1 no
    # tiene sentido. Se leen desde df, que siempre tiene las columnas físicas.
    fisicas = config["features"]   # las cinco físicas continuas (fijas)

    def resumen(sub_idx, etiqueta):
        filas = []
        for nv in ["bajo", "medio", "alto"]:
            m = (niveles == nv) & sub_idx
            if m.sum() == 0:
                continue
            fila = {"Grupo": etiqueta, "Nivel": nv, "n": int(m.sum())}
            # Rangos de variables físicas continuas (P10 – mediana – P90)
            for f in fisicas:
                fila[f"{f}_p10"] = round(df.loc[m, f].quantile(0.10), 2)
                fila[f"{f}_med"] = round(df.loc[m, f].median(), 2)
                fila[f"{f}_p90"] = round(df.loc[m, f].quantile(0.90), 2)
            # Conteo promedio de cada estadio (descriptivo, no del modelo)
            for est in config["estadios"]:
                fila[f"{est}_prom"] = round(df.loc[m, est].mean(), 2)
            filas.append(fila)
        return filas

    todo = pd.Series(True, index=y.index)
    perfil = resumen(todo, "GLOBAL")
    for reg in region.unique():
        perfil += resumen(region == reg, reg)
    return perfil, (p33, p66)


# ============================================================================
# BLOQUE 7 — GRÁFICOS
# ============================================================================
def graficos(config, shap_values, X_test, features, shap_reg):
    os.makedirs(config["carpeta_img"], exist_ok=True)
    suf = f"{config['target']}_{'conEst' if config['usar_estacion'] else 'sinEst'}"

    plt.figure()
    shap.summary_plot(shap_values, X_test, show=False)
    plt.tight_layout()
    plt.savefig(f"{config['carpeta_img']}/beeswarm_{suf}.png", dpi=150,
                bbox_inches="tight"); plt.close()

    if shap_reg:
        regiones = list(shap_reg.keys())
        x = np.arange(len(features))
        w = 0.8 / len(regiones)
        plt.figure(figsize=(9, 5))
        for i, reg in enumerate(regiones):
            plt.barh(x + i*w, shap_reg[reg], height=w, label=reg)
        plt.yticks(x + w*(len(regiones)-1)/2, features)
        plt.xlabel("SHAP medio absoluto")
        plt.title(f"SHAP por región — {suf}")
        plt.legend(); plt.tight_layout()
        plt.savefig(f"{config['carpeta_img']}/region_{suf}.png", dpi=150,
                    bbox_inches="tight"); plt.close()
    print(f"  Gráficos en: {config['carpeta_img']}/")


# ============================================================================
# BLOQUE 8 — GUARDADO EN EXCEL
# ============================================================================
def guardar_excel(config, shap_reg, features, perfil, tiempo_fit):
    ruta = config["excel_salida"]
    if not os.path.exists(ruta):
        wb = Workbook(); wb.active.title = "Ejecuciones"; wb.save(ruta)
    wb = load_workbook(ruta)
    fecha = datetime.now().strftime("%Y-%m-%d %H:%M")
    est_txt = "Sí" if config["usar_estacion"] else "No"
    fill = PatternFill("solid", start_color="2E75B6")
    font = Font(bold=True, color="FFFFFF")

    # Hoja de tiempos de entrenamiento (solo el .fit del modelo óptimo)
    if "Tiempos" not in wb.sheetnames:
        wst = wb.create_sheet("Tiempos")
        wst.append(["Fecha", "Modelo", "Target", "Estacion",
                    "Tiempo_fit_seg"])
        for c in wst[1]: c.fill = fill; c.font = font
    wst = wb["Tiempos"]
    wst.append([fecha, "RandomForest", config["target"], est_txt,
                round(tiempo_fit, 4)])

    # SHAP por región
    if "SHAP_Region" not in wb.sheetnames:
        ws = wb.create_sheet("SHAP_Region")
        ws.append(["Fecha", "Target", "Estacion", "Region", "Variable",
                   "SHAP_medio_abs", "Ranking"])
        for c in ws[1]: c.fill = fill; c.font = font
    ws = wb["SHAP_Region"]
    for reg, imp in shap_reg.items():
        for rank, idx in enumerate(np.argsort(imp)[::-1], 1):
            ws.append([fecha, config["target"], est_txt, reg, features[idx],
                       round(float(imp[idx]), 5), rank])

    # Perfiles de infestación (con estadios)
    if "Perfiles_Infestacion" not in wb.sheetnames:
        ws2 = wb.create_sheet("Perfiles_Infestacion")
        cab = ["Fecha", "Target", "Grupo", "Nivel", "n"]
        for f in config["features"]:
            cab += [f"{f}_p10", f"{f}_med", f"{f}_p90"]
        for est in config["estadios"]:
            cab += [f"{est}_prom"]
        ws2.append(cab)
        for c in ws2[1]: c.fill = fill; c.font = font
    ws2 = wb["Perfiles_Infestacion"]
    for fila in perfil:
        row = [fecha, config["target"], fila["Grupo"], fila["Nivel"], fila["n"]]
        for f in config["features"]:
            row += [fila[f"{f}_p10"], fila[f"{f}_med"], fila[f"{f}_p90"]]
        for est in config["estadios"]:
            row += [fila[f"{est}_prom"]]
        ws2.append(row)

    wb.save(ruta)
    print(f"  Registrado en Excel: hojas SHAP_Region y Perfiles_Infestacion")


# ============================================================================
# BLOQUE 9 — MAIN
# ============================================================================
def main():
    print("=" * 70)
    print(f"  SHAP REGIÓN + PERFILES — RF — {CONFIG['target']} — "
          f"estación: {'Sí' if CONFIG['usar_estacion'] else 'No'}")
    print("=" * 70)

    X, y, region, df, features = cargar_datos(CONFIG)
    modelo, X_test, y_test, region_te, tiempo_fit = entrenar(CONFIG, X, y, region)

    explainer, shap_values, importancia, direcciones = shap_global(
        modelo, X_test, features)
    print("\n  SHAP GLOBAL (ranking):")
    for rank, idx in enumerate(np.argsort(importancia)[::-1], 1):
        print(f"    {rank}. {features[idx]:<24} "
              f"{importancia[idx]:.4f} ({direcciones[idx]})")

    shap_reg = shap_por_region(explainer, X_test, region_te, features)
    print("\n  SHAP POR REGIÓN:")
    for reg, imp in shap_reg.items():
        orden = np.argsort(imp)[::-1]
        print(f"    {reg}:")
        for rank, idx in enumerate(orden, 1):
            print(f"        {rank}. {features[idx]:<22} {imp[idx]:.4f}")

    perfil, (p33, p66) = perfiles_infestacion(CONFIG, X, y, region, df)
    print(f"\n  PERFILES (bajo<={p33:.2f}, medio<={p66:.2f}, alto>{p66:.2f}):")
    print("  Estadios promedio por nivel (ej. región Magallanes):")
    for fila in perfil:
        if fila["Grupo"] == "Magallanes":
            ests = " ".join(f"{e}={fila[f'{e}_prom']}" for e in CONFIG["estadios"])
            print(f"    {fila['Nivel']:<6} n={fila['n']:<5} {ests}")

    graficos(CONFIG, shap_values, X_test, features, shap_reg)
    guardar_excel(CONFIG, shap_reg, features, perfil, tiempo_fit)
    print("\n  Completado.")


if __name__ == "__main__":
    main()

  SHAP REGIÓN + PERFILES — RF — Total_Parasitos_ — estación: No
  Hiperparámetros: {'max_depth': None, 'max_features': 1.0, 'min_samples_split': 2, 'n_estimators': 500}
